# geneML on Google Colab

Fungal gene prediction with **geneML** (https://github.com/hexagonbio/geneML).
Input: a genome FASTA (`.fasta`, `.fa`, `.fna`, also gzipped `.gz`).
Output: GFF3 + CDS FASTA + protein FASTA, downloaded as one zip file.

## How to use

1. `Runtime` -> `Change runtime type` -> select **T4 GPU**.
2. Run **Cell 1** (setup), once per session.
3. Run **Cell 2** (run). It asks you to upload your genome, then does everything
   through to downloading the results.

To annotate another genome in the same session, just run Cell 2 again.

In [ ]:
#@title Cell 1. Setup: install geneML and check the GPU (run once per session) { display-mode: "form" }

INSTALL_METHOD = "github"  #@param ["github", "pypi"]

import os
import subprocess
import sys

# Let several worker processes share one GPU instead of the first one
# reserving almost all of its memory. No effect on the numbers produced.
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

pkg = ("git+https://github.com/hexagonbio/geneML.git"
       if INSTALL_METHOD == "github" else "geneml")
print(f"Installing {pkg} ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print()
subprocess.run(["geneml", "--version"], check=False)

# Verify that TensorFlow really sees the GPU. If it does not, geneML silently
# falls back to the CPU and becomes roughly an order of magnitude slower.
gpu_check = subprocess.run(
    [sys.executable, "-c",
     "import os;os.environ['TF_CPP_MIN_LOG_LEVEL']='3';"
     "import tensorflow as tf;"
     "print('TF', tf.__version__);"
     "print('GPUs', tf.config.list_physical_devices('GPU'))"],
    text=True, capture_output=True, check=False)
print(gpu_check.stdout.strip())

if "PhysicalDevice" in gpu_check.stdout:
    print("\nSetup OK. TensorFlow can use the GPU. Continue with Cell 2.")
else:
    print("\nWARNING: TensorFlow cannot see a GPU.")
    print("Go to Runtime -> Change runtime type and select a GPU (T4),")
    print("then run this cell again. Running on CPU works but is far slower.")
    if gpu_check.stderr:
        print("\n" + gpu_check.stderr.strip()[-2000:])

In [ ]:
#@title Cell 2. Upload a genome, run geneML, and download the results { display-mode: "form" }

#@markdown Leave **FASTA_PATH** empty to upload a file from your computer.
#@markdown Fill it in to reuse a file already in this session, e.g. `/content/input/genome.fna.gz`.
FASTA_PATH = ""  #@param {type:"string"}
OUTPUT_PREFIX = "genome"  #@param {type:"string"}
GENE_ID_PREFIX = "geneML"  #@param {type:"string"}
#@markdown `MAX_TRANSCRIPTS = 1` reports only the primary transcript. Set it to 5 (the geneML default) to also get alternative isoforms, labelled in the GFF3 as `TranscriptVariant=INTRON_RETENTION`, `EXON_SKIPPING`, etc. Values above 5 have no extra effect.
MAX_TRANSCRIPTS = 1  #@param {type:"integer"}
MIN_GENE_SCORE = "dynamic"  #@param ["dynamic", "0.3", "0.4", "0.5", "0.6", "0.7"] {allow-input: true}
CONTIGS_FILTER = ""  #@param {type:"string"}
CORES = 2  #@param {type:"integer"}
USE_CPU_ONLY = False  #@param {type:"boolean"}
DOWNLOAD_RESULTS = True  #@param {type:"boolean"}
#@markdown Any other geneML flag can be passed through here, e.g. `--max-intron-size 1000 --min-exon-size 3`. Run `!geneml --help` in a new cell for the full list.
EXTRA_ARGS = ""  #@param {type:"string"}

import os
import shlex
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

LAUNCHER_SOURCE = r'''
# Thin launcher around the geneml command line.
#
# It exists for one reason: geneML 1.1.0 fails with
# "AttributeError: 'float' object has no attribute 'lower'" whenever
# --min-gene-score is given a number instead of "dynamic", because args.py
# parses the value into a float and params.py then calls .lower() on it. The
# value is handed back as a string so it reaches geneML unchanged. With
# "dynamic" nothing happens, and if the fix cannot be applied the run continues
# with geneML's own behaviour.
import sys

try:
    import geneml.params as _params

    _orig_build = _params.build_params_namedtuple

    def _build(args):
        if not isinstance(args.min_gene_score, str):
            args.min_gene_score = str(float(args.min_gene_score))
        return _orig_build(args)

    _params.build_params_namedtuple = _build
except Exception as exc:
    print("launcher: min-gene-score compatibility fix skipped (%s)" % exc,
          file=sys.stderr)

# geneml.main parses sys.argv at import time, so this must come last.
from geneml.main import main

main()
'''

INPUT_DIR = Path("/content/input")
PREPARED_DIR = Path("/content/input_prepared")
OUTPUT_DIR = Path("/content/geneml_output")
for d in (INPUT_DIR, PREPARED_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ 1. input
if FASTA_PATH.strip():
    genome_path = Path(FASTA_PATH.strip())
    if not genome_path.exists():
        raise FileNotFoundError(f"FASTA_PATH does not exist: {genome_path}")
else:
    from google.colab import files
    print("Select your genome FASTA (.fasta / .fa / .fna, optionally .gz) ...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    name = next(iter(uploaded))
    genome_path = INPUT_DIR / Path(name).name
    if Path(name).resolve() != genome_path.resolve():
        shutil.move(name, genome_path)

print("Input:", genome_path)

# ------------------------------------------------------------ 2. gunzip .gz
if genome_path.name.endswith(".gz"):
    plain_path = PREPARED_DIR / genome_path.name[:-3]
    if plain_path.exists() and plain_path.stat().st_size > 0:
        print("Reusing uncompressed copy:", plain_path)
    else:
        print("Decompressing to:", plain_path)
        with open(plain_path, "wb") as fh:
            subprocess.run(["gunzip", "-c", str(genome_path)], stdout=fh, check=True)
else:
    plain_path = genome_path

genome_bp = sum(len(line.strip()) for line in open(plain_path)
                if not line.startswith(">"))
print(f"Sequence to annotate: {genome_bp:,} bp")

# ------------------------------------------------------------- 3. command
out_base = OUTPUT_DIR / OUTPUT_PREFIX
OUTPUT_GFF3 = f"{out_base}.gff3"
OUTPUT_GENES = f"{out_base}.genes.fna"
OUTPUT_PROTEINS = f"{out_base}.proteins.faa"
OUTPUT_LOG = f"{out_base}.log"

launcher = Path("/content/_geneml_launcher.py")
launcher.write_text(LAUNCHER_SOURCE)

env = dict(os.environ)
# Let the worker processes share one GPU instead of the first one reserving
# nearly all of its memory. Does not affect the numbers that come out.
env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

cmd = [sys.executable, str(launcher),
       "-o", OUTPUT_GFF3,
       "-g", OUTPUT_GENES,
       "-p", OUTPUT_PROTEINS,
       "-c", str(CORES),
       "--max-transcripts", str(MAX_TRANSCRIPTS),
       "--min-gene-score", str(MIN_GENE_SCORE),
       "-v"]
if USE_CPU_ONLY:
    cmd.append("--cpu-only")
if GENE_ID_PREFIX.strip():
    cmd += ["--gene-id-prefix", GENE_ID_PREFIX.strip()]
if CONTIGS_FILTER.strip():
    cmd += ["--contigs-filter",
            ",".join(x.strip() for x in CONTIGS_FILTER.split(",") if x.strip())]
if EXTRA_ARGS.strip():
    cmd += shlex.split(EXTRA_ARGS.strip())
cmd.append(str(plain_path))

print("\ngeneml " + " ".join(shlex.quote(c) for c in cmd[2:]) + "\n")

# ----------------------------------------------------------------- 4. run
t0 = time.time()
subprocess.run(cmd, check=True, env=env)
elapsed = time.time() - t0
print(f"\ngeneML finished in {elapsed / 60:.1f} min "
      f"({genome_bp / elapsed:,.0f} bp/s)")

# ------------------------------------------------------------- 5. summary
import pandas as pd
from IPython.display import display

rows = []
mean_score = None
with open(OUTPUT_GFF3) as fh:
    for line in fh:
        if line.startswith("#"):
            if "geneml-mean-gene-score" in line:
                mean_score = line.split()[-1]
        elif line.strip():
            rows.append(line.rstrip("\n").split("\t"))

gff = pd.DataFrame(rows, columns=["contig", "source", "feature", "start", "end",
                                  "score", "strand", "phase", "attributes"])
if gff.empty:
    print("\nNo features were predicted.")
else:
    counts = gff["feature"].value_counts()
    print(f"\nGenes: {counts.get('gene', 0):,}   "
          f"mRNAs: {counts.get('mRNA', 0):,}   "
          f"CDS: {counts.get('CDS', 0):,}   "
          f"contigs with genes: {gff['contig'].nunique():,}")
    if mean_score:
        print(f"Mean gene score: {mean_score}")
    display(gff.head(10))

# -------------------------------------------------------- 6. zip, download
zip_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.geneml_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (OUTPUT_GFF3, OUTPUT_GENES, OUTPUT_PROTEINS, OUTPUT_LOG):
        if Path(p).exists():
            z.write(p, arcname=Path(p).name)
            print(f"{Path(p).name}: {Path(p).stat().st_size:,} bytes")

print("\nZip:", zip_path)
if DOWNLOAD_RESULTS:
    from google.colab import files as _files
    _files.download(str(zip_path))

## Notes

**Transcript variants.** With `MAX_TRANSCRIPTS = 1` each gene gets exactly one
mRNA, marked `TranscriptVariant=PRIMARY`. Set it to 5 to also report alternative
isoforms; geneML labels each extra mRNA with the splicing event that
distinguishes it from the primary one (`INTRON_RETENTION`, `EXON_SKIPPING`,
`ALT_FIRST_EXON`, `ALT_LAST_EXON`, `ALT_5_SPLICE_SITE`, `ALT_3_SPLICE_SITE`,
`COMPLEX`). The gene count is the same either way; only the number of mRNA
records changes. geneML caps the selection at 5, so larger values have no effect.

**Troubleshooting.**

- Out of memory: set `CORES = 1`.
- TensorFlow or GPU errors: set `USE_CPU_ONLY = True`. Much slower, same results.
- Timing a configuration: put a few contig IDs in `CONTIGS_FILTER`
  (comma separated). Use this for timing only, since `dynamic` scoring
  calibrates its threshold on the whole input.
- geneML GFF3 has no UTRs; CDS features match exon features apart from phase.